In [11]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
torch.backends.cudnn.enabled = False

BATCH_SIZE = 128
LEARNING_RATE = 0.001
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.2
WINDOW_SIZE = 30
EARLY_STOP_PATIENCE = 10
MODEL_SAVE_PATH = "../../models/multidim_LSTM.pth"
TRAIN_RATIO = 0.8
VAL_RATIO = 0.9


class MyLSTMSequential(nn.Module):

    lstm : nn.LSTM
    linear : nn.Linear
    activation : nn.Module
    lstm_cells : list[nn.LSTMCell]
    hidden_size : int
    num_layers : int

    def __init__(self, input_size: int, hidden_size: int, output_size: int, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, output_size)
        self.activation = nn.ReLU()
        self.lstm_cells = [nn.LSTMCell(input_size if i == 0 else hidden_size, hidden_size) for i in range(num_layers)]
        self.hidden_size = hidden_size
        self.num_layers = num_layers

    def forward(self, x):
        x, _ = self.lstm(x)
        x = self.activation(x[:, -1, :])
        x = self.linear(x)
        return x


def build_windows(data: np.ndarray, window_size: int) -> tuple[np.ndarray, np.ndarray]:
    """Build sliding-window (X, y) pairs. y is the value immediately after the window (one-step-ahead)."""
    x, y = [], []
    for i in range(len(data) - window_size):
        x.append(data[i : i + window_size])
        y.append(data[i + window_size])
    return (
        np.array(x),
        np.array(y),
    )

def get_daily_percentage_change(data: np.ndarray) -> np.ndarray:
    result = np.zeros(len(data))
    for i in range(1, len(data)):
        result[i] = ((data[i] - data[i - 1]) / data[i - 1]) * 10
    result[0] = 0
    return result

def get_device():
    if torch.cuda.is_available():
        print("Using CUDA")
        return torch.device("cuda")
    else:
        print("Using CPU")
        return torch.device("cpu")

def get_data():
    df_nsdq = pd.read_csv("../../datasets_aligned/NASDAQCOM.csv")
    df_cpi = pd.read_csv("../../datasets_aligned/CPIAUCSL.csv")
    df_gpr = pd.read_csv("../../datasets_aligned/data_gpr_daily_recent.CSV")
    df_interest = pd.read_csv("../../datasets_aligned/DFF_interest_rate.csv")
    df_gdp = pd.read_csv("../../datasets_aligned/GDPC1.csv")

    start_date = pd.Timestamp("1985-01-01")
    end_date = pd.Timestamp("2025-10-01")

    df_nsdq.index = pd.to_datetime(df_nsdq["date"])
    df_cpi.index = pd.to_datetime(df_cpi["date"])
    df_gpr.index = pd.to_datetime(df_gpr["date"])
    df_interest.index = pd.to_datetime(df_interest["date"])
    df_gdp.index = pd.to_datetime(df_gdp["date"])

    df_nsdq = df_nsdq.loc[start_date:end_date]
    df_cpi = df_cpi.loc[start_date:end_date]
    df_gpr = df_gpr.loc[start_date:end_date]
    df_interest = df_interest.loc[start_date:end_date]
    df_gdp = df_gdp.loc[start_date:end_date]

    data_nsdq = np.array(df_nsdq["NASDAQCOM"].values, dtype=np.float32)
    data_cpi = np.array(df_cpi["cpi"].values, dtype=np.float32)
    data_gpr = np.array(df_gpr["gpr"].values, dtype=np.float32)
    data_interest = np.array(df_interest["DFF"].values, dtype=np.float32)
    data_gdp = np.array(df_gdp["GDP"].values, dtype=np.float32)
    
    data_nsdq = get_daily_percentage_change(data_nsdq)
    data_cpi = get_daily_percentage_change(data_cpi)
    data_gpr = get_daily_percentage_change(data_gpr)
    data_interest = get_daily_percentage_change(data_interest)
    data_gdp = get_daily_percentage_change(data_gdp)

    return np.column_stack((data_nsdq, data_cpi, data_gpr, data_interest, data_gdp))


if __name__ == "__main__":


    device = get_device()

    data = get_data()



    print(data)
    print(data.shape)
    train_end = int(len(data) * TRAIN_RATIO)
    val_start = train_end - WINDOW_SIZE - 1
    val_end = int(len(data) * VAL_RATIO)
    test_start = val_end - WINDOW_SIZE - 1
    
    
    train_x, train_y = build_windows(data[:train_end], WINDOW_SIZE)
    val_x, val_y = build_windows(data[val_start:val_end], WINDOW_SIZE)
    test_x, test_y = build_windows(data[test_start:], WINDOW_SIZE)

    train_x = torch.tensor(train_x, dtype=torch.float32)
    train_y = torch.tensor(train_y, dtype=torch.float32)
    val_x = torch.tensor(val_x, dtype=torch.float32)
    val_y = torch.tensor(val_y, dtype=torch.float32)
    test_x = torch.tensor(test_x, dtype=torch.float32)
    test_y = torch.tensor(test_y, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(val_x, val_y),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    test_loader = DataLoader(
        TensorDataset(test_x, test_y),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = MyLSTMSequential(input_size=5, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT, output_size=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_model_state = None
    no_improve_count = 0
    epoch = 0
    while no_improve_count < EARLY_STOP_PATIENCE:
        # training
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            print(inputs.shape, targets.shape)
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        print(f"Epoch {epoch}, Train Loss: {train_loss}")

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        print(f"Epoch {epoch}, Validation Loss: {val_loss}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            no_improve_count = 0
        else:
            no_improve_count += 1
        epoch += 1

    print(f"training completed in {epoch} epochs")
    print(f"best validation loss: {best_val_loss}")
    print(f"best model saved to {MODEL_SAVE_PATH}")
    
    os.makedirs("models", exist_ok=True)
    torch.save(best_model_state, MODEL_SAVE_PATH)


Using CUDA
[[ 0.          0.          0.          0.          0.        ]
 [-0.02919358  0.         -2.48565435  0.72082394  0.        ]
 [ 0.02033264  0.         -1.45493436 -0.64034092  0.        ]
 ...
 [ 0.01582471  0.         -0.07008524  0.          0.        ]
 [ 0.03048069  0.          0.36898035  0.          0.        ]
 [ 0.04199045  0.          0.5040502   0.          0.01628225]]
(14884, 5)
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])


C:\Users\ton20\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([128, 5])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Siz

C:\Users\ton20\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([101, 5])) that is different to the input size (torch.Size([101, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, Train Loss: 0.1712075672963614


C:\Users\ton20\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([81, 5])) that is different to the input size (torch.Size([81, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, Validation Loss: 0.1709375809878111
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size([128, 5])
torch.Size([128, 30, 5]) torch.Size

KeyboardInterrupt: 